In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/manvidhamija/mail-spam/enron_spam_data.csv


In [2]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:10]:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/manvidhamija/mail-spam/enron_spam_data.csv


In [3]:
import pandas as pd

file_path = "/kaggle/input/datasets/manvidhamija/mail-spam/enron_spam_data.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset information:")
df.info()


Dataset shape: (33716, 5)

Column names:
['Message ID', 'Subject', 'Message', 'Spam/Ham', 'Date']

First 5 rows:


,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Message ID  33716 non-null  int64 
 1   Subject     33427 non-null  object
 2   Message     33345 non-null  object
 3   Spam/Ham    33716 non-null  object
 4   Date        33716 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


In [4]:

# 1. Check missing values
print("Missing values in each column:")
print(df.isnull().sum())

# 2. Check spam vs ham distribution
print("\nSpam/Ham distribution:")
print(df["Spam/Ham"].value_counts())

print("\nClass distribution in percentage:")
print(
    (df["Spam/Ham"].value_counts(normalize=True) * 100)
    .round(2)
)

# 3. Check duplicate records
print("\nDuplicate rows:", df.duplicated().sum())

# 4. Inspect examples from both classes
print("\nExample HAM email:")
display(df.loc[df["Spam/Ham"] == "ham", ["Subject", "Message"]].head(2))

print("\nExample SPAM email:")
display(df.loc[df["Spam/Ham"] == "spam", ["Subject", "Message"]].head(2))


Missing values in each column:
Message ID      0
Subject       289
Message       371
Spam/Ham        0
Date            0
dtype: int64

Spam/Ham distribution:
Spam/Ham
spam    17171
ham     16545
Name: count, dtype: int64

Class distribution in percentage:
Spam/Ham
spam    50.93
ham     49.07
Name: proportion, dtype: float64

Duplicate rows: 0

Example HAM email:


,Subject,Message
0,christmas tree farm pictures,NaN
1,"vastar resources , inc .","gary , production from the high island larger ..."



Example SPAM email:


,Subject,Message
3672,dobmeos with hgh my energy level has gone up !...,introducing\ndoctor - formulated\nhgh\nhuman g...
3673,your prescription is ready . . oxwq s f e,"low cost prescription medications\nsoma , ultr..."


In [5]:

# Keep only the columns needed for classification
data = df[["Subject", "Message", "Spam/Ham"]].copy()

# Replace missing text with empty strings
data["Subject"] = data["Subject"].fillna("")
data["Message"] = data["Message"].fillna("")

# Combine subject and body
data["Email_Text"] = (
    data["Subject"].str.strip() + " " +
    data["Message"].str.strip()
).str.strip()

# Remove emails with no usable text
before = len(data)
data = data[data["Email_Text"].str.len() > 0].copy()

# Normalize labels
data["Spam/Ham"] = data["Spam/Ham"].str.strip().str.lower()

# Keep only valid labels
data = data[data["Spam/Ham"].isin(["spam", "ham"])].copy()

print("Rows before cleaning:", before)
print("Rows after cleaning:", len(data))
print("Rows removed:", before - len(data))

print("\nFinal class distribution:")
print(data["Spam/Ham"].value_counts())

print("\nSample prepared emails:")
display(data[["Email_Text", "Spam/Ham"]].head())


Rows before cleaning: 33716
Rows after cleaning: 33665
Rows removed: 51

Final class distribution:
Spam/Ham
spam    17120
ham     16545
Name: count, dtype: int64

Sample prepared emails:


,Email_Text,Spam/Ham
0,christmas tree farm pictures,ham
1,"vastar resources , inc . gary , production fro...",ham
2,calpine daily gas nomination - calpine daily g...,ham
3,re : issue fyi - see note below - already done...,ham
4,meter 7268 nov allocation fyi .\n- - - - - - -...,ham


In [6]:
from sklearn.model_selection import train_test_split

# Input feature and target label
X = data["Email_Text"]
y = data["Spam/Ham"]

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

Training samples: 26932
Testing samples: 6733

Training label distribution:
Spam/Ham
spam    13696
ham     13236
Name: count, dtype: int64

Testing label distribution:
Spam/Ham
spam    3424
ham     3309
Name: count, dtype: int64


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Create the ML pipeline
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=50000
    )),
    ("naive_bayes", MultinomialNB(alpha=1.0))
])

# Train the model
model.fit(X_train, y_train)

print("Model training completed successfully!")

# Generate predictions on unseen test emails
y_pred = model.predict(X_test)

print("\nFirst 10 predictions:")
print(y_pred[:10])

Model training completed successfully!

First 10 predictions:
['ham' 'ham' 'ham' 'spam' 'ham' 'spam' 'ham' 'ham' 'ham' 'ham']


In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision (spam):", round(
    precision_score(y_test, y_pred, pos_label="spam"), 4
))
print("Recall (spam):", round(
    recall_score(y_test, y_pred, pos_label="spam"), 4
))
print("F1-score (spam):", round(
    f1_score(y_test, y_pred, pos_label="spam"), 4
))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=["ham", "spam"]))


Accuracy: 0.9883
Precision (spam): 0.9895
Recall (spam): 0.9874
F1-score (spam): 0.9885

Classification Report:
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99      3309
        spam       0.99      0.99      0.99      3424

    accuracy                           0.99      6733
   macro avg       0.99      0.99      0.99      6733
weighted avg       0.99      0.99      0.99      6733

Confusion Matrix:
[[3273   36]
 [  43 3381]]


In [9]:

# Test on newly written email examples
test_emails = [
    {
        "Subject": "Congratulations! You won a prize",
        "Message": "Claim your free cash reward now. Limited time offer!"
    },
    {
        "Subject": "Tomorrow's project meeting",
        "Message": "Hi team, please attend the project meeting tomorrow at 10 AM."
    },
    {
        "Subject": "Urgent account verification",
        "Message": "Verify your account immediately to claim your reward."
    },
    {
        "Subject": "Assignment submission reminder",
        "Message": "Please submit your assignment before the deadline."
    }
]

custom_df = pd.DataFrame(test_emails)

custom_df["Email_Text"] = (
    custom_df["Subject"] + " " + custom_df["Message"]
)

custom_df["Predicted_Label"] = model.predict(
    custom_df["Email_Text"]
)

display(
    custom_df[
        ["Subject", "Predicted_Label"]
    ]
)


,Subject,Predicted_Label
0,Congratulations! You won a prize,spam
1,Tomorrow's project meeting,ham
2,Urgent account verification,spam
3,Assignment submission reminder,ham


In [10]:
import joblib
import os

# Save the complete trained pipeline
model_path = "/kaggle/working/email_spam_nb_model.pkl"
joblib.dump(model, model_path)

print("Model saved successfully!")
print("File path:", model_path)
print("File size:", round(os.path.getsize(model_path) / 1024, 2), "KB")


Model saved successfully!
File path: /kaggle/working/email_spam_nb_model.pkl
File size: 3371.41 KB


In [11]:
# Load the saved model
loaded_model = joblib.load(
    "/kaggle/working/email_spam_nb_model.pkl"
)

# Test one new email
sample_email = [
    "Congratulations! You have won a free prize. Claim now!"
]

prediction = loaded_model.predict(sample_email)[0]
print("Predicted category:", prediction)


Predicted category: spam


In [12]:
# Check exact duplicate email texts across the full cleaned dataset
print("Duplicate email texts:", data["Email_Text"].duplicated().sum())

# Check exact duplicate email texts in training and testing sets
train_texts = set(X_train)
test_texts = set(X_test)

overlap = train_texts.intersection(test_texts)

print("Unique training texts:", len(train_texts))
print("Unique testing texts:", len(test_texts))
print("Exact text overlap between train and test:", len(overlap))

Duplicate email texts: 3203
Unique training texts: 24844
Unique testing texts: 6613
Exact text overlap between train and test: 995


In [13]:
# Check whether identical email texts have different labels
label_counts = data.groupby("Email_Text")["Spam/Ham"].nunique()

conflicting_texts = label_counts[label_counts > 1]

print("Unique email texts:", data["Email_Text"].nunique())
print("Texts with conflicting labels:", len(conflicting_texts))

if len(conflicting_texts) > 0:
    print("\nExamples of conflicting texts:")
    display(
        data[data["Email_Text"].isin(conflicting_texts.index)]
        [["Email_Text", "Spam/Ham"]]
        .head(10)
    )


Unique email texts: 30462
Texts with conflicting labels: 0


In [14]:

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Group identical email texts together
X_all = data["Email_Text"]
y_all = data["Spam/Ham"]
groups = data["Email_Text"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X_all, y_all, groups=groups)
)

X_train_group = X_all.iloc[train_idx]
X_test_group = X_all.iloc[test_idx]
y_train_group = y_all.iloc[train_idx]
y_test_group = y_all.iloc[test_idx]

print("Training samples:", len(X_train_group))
print("Testing samples:", len(X_test_group))
print(
    "Exact text overlap:",
    len(set(X_train_group).intersection(set(X_test_group)))
)

# Train a fresh pipeline on the grouped split
group_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=50000
    )),
    ("naive_bayes", MultinomialNB(alpha=1.0))
])

group_model.fit(X_train_group, y_train_group)
group_pred = group_model.predict(X_test_group)

print("\nGrouped-split accuracy:",
      round(accuracy_score(y_test_group, group_pred), 4))

print("\nClassification report:")
print(classification_report(y_test_group, group_pred))

print("\nConfusion matrix [ham, spam]:")
print(confusion_matrix(
    y_test_group, group_pred, labels=["ham", "spam"]
))


Training samples: 26920
Testing samples: 6745
Exact text overlap: 0

Grouped-split accuracy: 0.9852

Classification report:
              precision    recall  f1-score   support

         ham       0.99      0.98      0.99      3341
        spam       0.98      0.99      0.99      3404

    accuracy                           0.99      6745
   macro avg       0.99      0.99      0.99      6745
weighted avg       0.99      0.99      0.99      6745


Confusion matrix [ham, spam]:
[[3285   56]
 [  44 3360]]


In [15]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

group_pred = group_model.predict(X_test_group)

print("Grouped-split accuracy:",
      round(accuracy_score(y_test_group, group_pred), 4))

print("\nClassification report:")
print(classification_report(y_test_group, group_pred))

print("\nConfusion matrix [ham, spam]:")
print(confusion_matrix(
    y_test_group, group_pred, labels=["ham", "spam"]
))

Grouped-split accuracy: 0.9852

Classification report:
              precision    recall  f1-score   support

         ham       0.99      0.98      0.99      3341
        spam       0.98      0.99      0.99      3404

    accuracy                           0.99      6745
   macro avg       0.99      0.99      0.99      6745
weighted avg       0.99      0.99      0.99      6745


Confusion matrix [ham, spam]:
[[3285   56]
 [  44 3360]]
